In [ ]:
using Pkg
Pkg.add(["Images", "FileIO", "ImageIO", "PlotlyJS", "Printf"])

In [1]:
using Images, FileIO, Random, Statistics, Printf, PlotlyJS

IMG_SIZE = 150
OUTPUT_DIR = "lab19"
mkpath(OUTPUT_DIR)

function draw_line!(img::Matrix{Float64}, x1::Int, y1::Int, x2::Int, y2::Int; thick::Int=1)
    dx = abs(x2 - x1)
    dy = abs(y2 - y1)
    sx = x1 < x2 ? 1 : -1
    sy = y1 < y2 ? 1 : -1
    err = dx - dy
    while true
        for δx in -thick:thick, δy in -thick:thick
            px, py = x1 + δx, y1 + δy
            if 1 <= px <= IMG_SIZE && 1 <= py <= IMG_SIZE
                img[py, px] = 0.0
            end
        end
        x1 == x2 && y1 == y2 && break
        e2 = 2 * err
        if e2 > -dy; err -= dy; x1 += sx; end
        if e2 < dx; err += dx; y1 += sy; end
    end
end

function mutate_gene(genes::Vector{Int}, ranges::Vector{Tuple{Int,Int}})::Vector{Int}
    g = copy(genes)
    i = rand(1:length(g))
    g[i] += rand([-1, 1])
    g[i] = clamp(g[i], ranges[i][1], ranges[i][2])
    return g
end


function distance_transform(mask::BitMatrix)::Matrix{Float64}
    h, w = size(mask)
    dist = fill(Inf, h, w)
    queue = Tuple{Int,Int}[]
    for y in 1:h, x in 1:w
        if mask[y, x]
            dist[y, x] = 0.0
            push!(queue, (y, x))
        end
    end
    head = 1
    while head <= length(queue)
        cy, cx = queue[head]
        head += 1
        for dy in -1:1, dx in -1:1
            (dy == 0 && dx == 0) && continue
            ny, nx = cy + dy, cx + dx
            (1 <= ny <= h && 1 <= nx <= w) || continue
            nd = dist[cy, cx] + sqrt(Float64(dy^2 + dx^2))
            if nd < dist[ny, nx]
                dist[ny, nx] = nd
                push!(queue, (ny, nx))
            end
        end
    end
    return dist
end

function combined_distance(a::Matrix{Float64}, b::Matrix{Float64}; alpha::Float64=0.65)::Float64
    a_black = a .< 0.5
    b_black = b .< 0.5
    na, nb = sum(a_black), sum(b_black)
    manhattan = sum(abs.(a .- b)) / length(a)
    if na == 0 || nb == 0
        return manhattan * 100.0
    end
    dt_a = distance_transform(a_black)
    dt_b = distance_transform(b_black)
    chamfer = (mean(dt_a[b_black]) + mean(dt_b[a_black])) / 2.0
    return alpha * chamfer + (1.0 - alpha) * manhattan * 100.0
end


TARGET_PATH = "target_edges.png"
target_raw = load(TARGET_PATH)
target = Float64.(Gray.(imresize(target_raw, (IMG_SIZE, IMG_SIZE))))
save(joinpath(OUTPUT_DIR, "target.png"), Gray.(target))
println("Цель: $(size(target)), чёрных пикселей: $(@sprintf("%.1f", 100*(1-mean(target))))%")


function make_stems(g::AbstractVector{Int})
    return [
        (0, g[1]), (g[2], g[3]), (g[4], 0), (g[5], -g[6]),
        (0, -g[7]), (-g[5], -g[6]), (-g[4], 0), (-g[2], g[3])
    ]
end

function tree_render(stems, len::Int, dir::Int, pos::Tuple{Float64,Float64})
    segs = Tuple{Tuple{Float64,Float64}, Tuple{Float64,Float64}}[]
    nd = mod(dir, 8) + 1
    dx, dy = stems[nd]
    np_ = (pos[1] + len * dx, pos[2] + len * dy)
    push!(segs, (pos, np_))
    if len > 1
        append!(segs, tree_render(stems, len - 1, dir + 1, np_))
        append!(segs, tree_render(stems, len - 1, dir - 1, np_))
    end
    return segs
end

function segs_to_image(segs; cx_off::Int=0, cy_off::Int=0,
                       scale_mult::Float64=1.0, line_thick::Int=1)::Matrix{Float64}
    isempty(segs) && return ones(IMG_SIZE, IMG_SIZE)
    ax = Float64[]; ay = Float64[]
    for (s, f) in segs
        push!(ax, s[1], f[1]); push!(ay, s[2], f[2])
    end
    w = max(abs(minimum(ax)), maximum(ax)) * 2
    h = max(abs(minimum(ay)), maximum(ay)) * 2
    max(w, h) == 0 && return ones(IMG_SIZE, IMG_SIZE)
    factor = (IMG_SIZE - 10) / max(w, h) * scale_mult
    cx = IMG_SIZE ÷ 2 + cx_off
    cy = IMG_SIZE ÷ 2 + cy_off
    img = ones(IMG_SIZE, IMG_SIZE)
    for (s, f) in segs
        draw_line!(img,
            round(Int, s[1]*factor + cx), round(Int, -s[2]*factor + cy),
            round(Int, f[1]*factor + cx), round(Int, -f[2]*factor + cy); thick=line_thick)
    end
    return img
end


CLASSIC_RANGES = Tuple{Int,Int}[
    (-9,9), (-9,9), (-9,9), (-9,9), (-9,9), (-9,9), (-9,9), (-9,9), (2,7)
]

function classic_draw(genes::Vector{Int})::Matrix{Float64}
    stems = make_stems(genes[1:8])
    segs = try
        tree_render(stems, genes[9], 0, (0.0, 0.0))
    catch e
        e isa StackOverflowError ? Tuple{Tuple{Float64,Float64},Tuple{Float64,Float64}}[] : rethrow()
    end
    return segs_to_image(segs; line_thick=1)
end


N_TREES = 4

NEO_RANGES = Tuple{Int,Int}[
    (-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(2,6),
    (-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(2,6),
    (-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(2,6),
    (-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(-9,9),(2,6),
    (-9,9),(-9,9),(-9,9),(-9,9),
    (-9,9),(-9,9),(-9,9),(-9,9),
    (1,5),(1,5),(1,5),(1,5)
]

function neo_draw(genes::Vector{Int})::Matrix{Float64}
    img = ones(IMG_SIZE, IMG_SIZE)
    for t in 1:N_TREES
        base = (t - 1) * 9
        tg = genes[base+1:base+9]
        off_base = N_TREES * 9
        ox = genes[off_base + t]
        oy = genes[off_base + N_TREES + t]
        sc = genes[off_base + 2*N_TREES + t]
        stems = make_stems(tg[1:8])
        segs = try
            tree_render(stems, tg[9], 0, (0.0, 0.0))
        catch e
            e isa StackOverflowError ? Tuple{Tuple{Float64,Float64},Tuple{Float64,Float64}}[] : rethrow()
        end
        isempty(segs) && continue
        ax = [p[1] for s in segs for p in s]
        ay = [p[2] for s in segs for p in s]
        w = max(abs(minimum(ax)), maximum(ax)) * 2
        h = max(abs(minimum(ay)), maximum(ay)) * 2
        max(w, h) == 0 && continue
        factor = (IMG_SIZE - 10) / max(w, h) * (sc / 3.0)
        cx = IMG_SIZE ÷ 2 + ox * 5
        cy = IMG_SIZE ÷ 2 + oy * 5
        for (s, f) in segs
            draw_line!(img,
                round(Int, s[1]*factor + cx), round(Int, -s[2]*factor + cy),
                round(Int, f[1]*factor + cx), round(Int, -f[2]*factor + cy); thick=0)
        end
    end
    return img
end


function evolve(draw_fn::Function, ranges::Vector{Tuple{Int,Int}},
                target::Matrix{Float64};
                n_offspring::Int=80, stag_limit::Int=80,
                max_gen::Int=1500, seed::Int=42)
    Random.seed!(seed)
    parent = [rand(lo:hi) for (lo, hi) in ranges]
    parent_img = draw_fn(parent)
    best_dist = combined_distance(parent_img, target)
    history = Float64[best_dist]
    snapshots = Pair{Int, Matrix{Float64}}[0 => copy(parent_img)]
    gen = 0
    stag = 0
    while stag < stag_limit && gen < max_gen
        gen += 1
        improved = false
        for _ in 1:n_offspring
            child = mutate_gene(parent, ranges)
            child_img = draw_fn(child)
            dist = combined_distance(child_img, target)
            if dist < best_dist
                best_dist = dist
                parent = child
                parent_img = child_img
                improved = true
            end
        end
        improved ? (stag = 0) : (stag += 1)
        push!(history, best_dist)
        if gen <= 15 || gen % 25 == 0
            push!(snapshots, gen => copy(parent_img))
        end
        if gen % 100 == 0
            println("  Gen $gen | Dist: $(@sprintf("%.3f", best_dist)) | Stag: $stag")
        end
    end
    push!(snapshots, gen => copy(parent_img))
    return parent, parent_img, history, snapshots
end


function side_by_side(biomorph::Matrix{Float64}, tgt::Matrix{Float64})
    hcat(tgt, ones(IMG_SIZE, 5), biomorph)
end

function save_experiment(prefix::String, result, tgt::Matrix{Float64})
    genes, img, hist, snaps = result
    save(joinpath(OUTPUT_DIR, "$(prefix)_comparison.png"), Gray.(side_by_side(img, tgt)))

    n = min(16, length(snaps))
    idxs = round.(Int, range(1, length(snaps), length=n))
    grid = ones(4*(IMG_SIZE+5)-5, 4*(IMG_SIZE+5)-5)
    for (k, i) in enumerate(idxs)
        r, c = (k-1) ÷ 4, (k-1) % 4
        grid[r*(IMG_SIZE+5)+1 : r*(IMG_SIZE+5)+IMG_SIZE,
             c*(IMG_SIZE+5)+1 : c*(IMG_SIZE+5)+IMG_SIZE] .= snaps[i].second
    end
    save(joinpath(OUTPUT_DIR, "$(prefix)_grid.png"), Gray.(grid))

    gif_frames = [Gray.(side_by_side(s.second, tgt)) for s in snaps]
    save(joinpath(OUTPUT_DIR, "$(prefix)_evolution.gif"), cat(gif_frames...; dims=3); fps=5)

    println("Сохранено: $(prefix)_comparison/grid/evolution")
end


println("=" ^ 60)
println("ЭКСПЕРИМЕНТ 1: Классика (9 генов)")
println("=" ^ 60)

best_classic = nothing
for run in 0:2
    genes, img, hist, snaps = evolve(classic_draw, CLASSIC_RANGES, target;
                                      n_offspring=80, stag_limit=80, max_gen=1000, seed=run*111)
    println("Run $run: gen=$(length(hist)-1), dist=$(@sprintf("%.3f", hist[end])), genes=$genes")
    if best_classic === nothing || hist[end] < best_classic[3][end]
        best_classic = (genes, img, hist, snaps)
    end
end
println("Лучший: dist=$(@sprintf("%.3f", best_classic[3][end]))")
save_experiment("classic", best_classic, target)


println("\n" * "=" ^ 60)
println("ЭКСПЕРИМЕНТ 2: Неоклассика (48 генов, 4 дерева)")
println("=" ^ 60)

best_neo = nothing
for run in 0:2
    genes, img, hist, snaps = evolve(neo_draw, NEO_RANGES, target;
                                      n_offspring=100, stag_limit=80, max_gen=1000, seed=run*222+7)
    println("Run $run: gen=$(length(hist)-1), dist=$(@sprintf("%.3f", hist[end]))")
    if best_neo === nothing || hist[end] < best_neo[3][end]
        best_neo = (genes, img, hist, snaps)
    end
end
println("Лучший: dist=$(@sprintf("%.3f", best_neo[3][end]))")
save_experiment("neo", best_neo, target)


p = plot([
    scatter(x=0:length(best_classic[3])-1, y=best_classic[3],
            mode="lines", name="Классика (9 генов)",
            line=attr(color="royalblue", width=2)),
    scatter(x=0:length(best_neo[3])-1, y=best_neo[3],
            mode="lines", name="Неоклассика (48 генов)",
            line=attr(color="crimson", width=2))
], Layout(
    title="Сходимость эволюции биоморф",
    xaxis_title="Поколение",
    yaxis_title="Расстояние (Chamfer + Manhattan)",
    template="plotly_white",
    width=900, height=500,
    legend=attr(x=0.6, y=0.95)
))
savefig(p, joinpath(OUTPUT_DIR, "convergence.html"))
display(p)

println("\nРезультаты в: $OUTPUT_DIR/")

WebIO._IJuliaInit()

Цель: (150, 150), чёрных пикселей: 6.8%
ЭКСПЕРИМЕНТ 1: Классика (9 генов)
Run 0: gen=82, dist=7.694, genes=[-3, -2, 9, -8, 2, -6, -7, 4, 2]
Run 1: gen=81, dist=7.639, genes=[1, -1, -4, -6, -2, 1, -2, 7, 2]
Run 2: gen=82, dist=7.123, genes=[-2, 1, 4, -3, 6, 0, -5, -9, 4]
Лучший: dist=7.123
Сохранено: classic_comparison/grid/evolution

ЭКСПЕРИМЕНТ 2: Неоклассика (48 генов, 4 дерева)
Run 0: gen=86, dist=6.508
Run 1: gen=87, dist=5.366
Run 2: gen=89, dist=6.018
Лучший: dist=5.366
Сохранено: neo_comparison/grid/evolution


data: [
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields height, legend, margin, template, title, width, xaxis, and yaxis"


Результаты в: lab19/
